# 📅 2026-05-25 (월) 개발 노트
## Phase 1.5 완료 + Phase 2 (Google OAuth2) 시작

## 🎯 오늘의 목표
- [x] Phase 1.5: POST /taste/action 엔드포인트 (행동 로그 수집)
- [x] 프론트 행동 로그 연동 (GameCard, GameDetail, 메인, 취향 분석)
- [x] DEFAULT_PREFERENCES 버그 수정 (추천 품질 개선)
- [x] Phase 2: Google OAuth2 로그인 구현
- [x] DB 백업 스크립트 + Task Scheduler 등록
- [x] scripts/ 폴더 구조화
- [x] 0522~0524 개발 노트 작성
- [x] npm run build 통과

---

## 🛠 Phase 1.5: 행동 로그 수집 인프라

### 백엔드

#### FastAPI ORM 모델 추가
```python
# fastapi_app/models/user_action.py (신규)
class UserAction(Base):
    __tablename__ = 'user_actions'  # Django 테이블 공유
    id          = Column(BigInteger, primary_key=True)
    user_id     = Column(Integer, nullable=True)
    session_id  = Column(String(64), nullable=False)
    app_id      = Column(Integer, nullable=True)
    action_type = Column(String(30), nullable=False)
    context     = Column(JSONB, default=dict)
    created_at  = Column(DateTime(timezone=True), nullable=False)  # Django DEFAULT 없음 → Python 직접 주입
```

#### Taste 라우터
```python
# fastapi_app/routers/taste.py (신규)
# POST /api/v1/taste/action   → 행동 로그 기록 (Rate Limit 60/min)
# GET  /api/v1/taste/stats    → 통계 (Admin Basic Auth)

# 보안 개선 사항 (오퍼스 리뷰 반영):
# - session_id: min_length 16, 영숫자만 (field_validator)
# - action_type: Literal 타입 (Pydantic 422 자동 처리)
# - context: 8KB 제한
# - /stats: Basic Auth 보호
# - Rate Limit: 60/minute
# - Raw SQL → SQLAlchemy ORM 전환
# - Fire-and-forget: 실패해도 200 반환 (서비스 영향 없음)
```

#### config.py 추가
```python
RATE_LIMIT_TASTE: str = '60/minute'
ADMIN_USERNAME: str = ''
ADMIN_PASSWORD: str = ''
APP_VERSION: str = '3.2.0'
```

### 프론트엔드

#### api.ts 행동 로그 함수 추가
```typescript
// 일반 비동기 (페이지 머무를 때)
export async function recordTasteAction(payload): Promise<TasteActionResponse | null>

// sendBeacon (navigation 시에도 전송 보장 — Steam 클릭)
export function recordTasteActionBeacon(payload): boolean
```

#### useUserStore sessionId 추가
```typescript
// SSR-safe lazy initialization
sessionId: null,  // 초기값 null → onRehydrateStorage에서 생성
ensureSessionId: () => string  // 없으면 생성 후 저장

// crypto.randomUUID() 우선, fallback은 수동 생성
```

#### 행동 로그 수집 커버리지
```
/ (메인)       → search  (시맨틱 검색 제출 시)
/search        → search  (취향 분석 제출 시)
/game/[appId]  → detail_view (진입 시, useRef 중복 방지)
               → steam_click (Steam 버튼, sendBeacon 사용)
GameCard       → rec_click / search_click (context 기반 구분)
               context: { click_position, displayed_score, referrer_context }
```

### 트러블슈팅
```
문제: INSERT INTO user_actions → created_at null 위반
원인: Django 테이블에 DEFAULT now() 없음 (앱 레벨 관리)
해결: UserAction 생성 시 created_at=datetime.now(timezone.utc) 직접 주입

문제: useEffect에서 detail_view 두 번 기록 (StrictMode)
원인: sessionId 의존성 + StrictMode 이중 실행
해결: useRef(false)로 중복 방지 + 100ms 디바운스
      sessionId 의존성 제거 → ensureSessionId() 직접 호출

문제: Steam 클릭 시 행동 로그 손실
원인: 새 탭 열기와 동시에 요청 취소
해결: navigator.sendBeacon() 사용 → 페이지 이탈 후에도 전송 보장
```

### 테스트 결과
```bash
curl -X POST http://127.0.0.1:8000/api/v1/taste/action \
  -d '{"session_id":"test1234567890abcd","app_id":1086940,"action_type":"detail_view"}'
# → {"success":true,"action_id":2}

curl http://127.0.0.1:8000/api/v1/taste/stats -u 'admin:비밀번호'
# → {"total_actions":1,"by_type":[{"action_type":"detail_view","count":1}]}
```

---

## 🔴 DEFAULT_PREFERENCES 버그 수정

### 문제
```
증상: 메인 페이지 추천 결과가 Spider-Man, GTA, 배트맨 등 액션 어드벤처에 편중
기대: narrative_depth: 8 설정 시 Disco Elysium, Planescape, Witcher 등 나와야 함

원인: DEFAULT_PREFERENCES 6개 지표 분산
  narrative_depth: 8
  lore_richness: 7
  art_style_uniqueness: 7  ← 액션 게임에도 높음
  replay_value: 6           ← 액션 게임 강점
  exploration_reward: 7     ← 어드벤처 강점
  soundtrack_impact: 7
  → 6개 모두 primary 처리 → 액션-어드벤처에 유리
```

### 검증
```javascript
// 콘솔 테스트: narrative_depth: 10 단독
fetch('/api/v1/games/recommend/by-preference', {
  method: 'POST',
  body: JSON.stringify({ preferences: { narrative_depth: 10 }, count: 10 })
})
// 결과: Trails beyond Horizon, FF7 Remake, Witcher, Planescape, Baldur's Gate 3
// → 백엔드 정상, 프론트 DEFAULT_PREFERENCES 문제
```

### 해결: 매일 순환 테마 시스템
```typescript
// src/hooks/useRecommend.ts
const DAILY_THEMES = [
  { name: 'narrative',    label: '서사 깊은',   prefs: { narrative_depth: 10, lore_richness: 9, choice_consequence: 8 } },
  { name: 'cozy',         label: '아늑한',      prefs: { cozy_factor: 9, time_pressure: 1, humor_rating: 7 } },
  { name: 'strategic',    label: '뇌지컬',      prefs: { strategic_depth: 10, management_complexity: 8, learning_curve: 7 } },
  { name: 'atmospheric',  label: '분위기 깊은', prefs: { environmental_storytelling: 9, soundtrack_impact: 9, melancholy: 7 } },
  { name: 'hidden_gem',   label: '숨겨진 보석', prefs: { art_style_uniqueness: 9, audio_design: 8, replay_value: 8 } },
]

// 날짜 기반 자동 순환
const dayOfYear = Math.floor((Date.now() - new Date(year, 0, 0)) / 86400000)
todaysTheme = DAILY_THEMES[dayOfYear % DAILY_THEMES.length]

// 효과:
// - 지표 3개로 압축 → 변별력 강화
// - 매일 다른 테마 → 신선한 추천
// - '오늘의 AI 추천 · 서사 깊은 게임' 라벨 표시
```

---

## 🔐 Phase 2: Google OAuth2 로그인

### 백엔드 설정
```python
# django_core/requirements.txt 추가
django-allauth>=65.0.0
djangorestframework-simplejwt>=5.3.0

# settings.py 추가
INSTALLED_APPS += ['allauth', 'allauth.account', 'allauth.socialaccount',
                   'allauth.socialaccount.providers.google', 'rest_framework_simplejwt']

SIMPLE_JWT = {
    'ACCESS_TOKEN_LIFETIME':  timedelta(hours=1),
    'REFRESH_TOKEN_LIFETIME': timedelta(days=30),
    'ROTATE_REFRESH_TOKENS':  True,
}

SOCIALACCOUNT_PROVIDERS = {
    'google': {
        'APP': { 'client_id': GOOGLE_CLIENT_ID, 'secret': GOOGLE_CLIENT_SECRET },
        'SCOPE': ['profile', 'email'],
    }
}
```

### 인증 흐름
```
1. 프론트 → /accounts/google/login/ (allauth)
2. Google 인증
3. Django → /api/auth/google/callback/
   - JWT 발급 (access 1h, refresh 30d)
   - 닉네임 없으면 Google 이름으로 자동 설정
4. 프론트 /auth/callback?access=...&refresh=... 리다이렉트
5. 프론트에서 토큰 저장 + /api/auth/me/ 유저 정보 조회
6. useUserStore에 user, accessToken, refreshToken 저장
7. 메인 페이지로 이동
```

### 신규 파일
```
django_core/apps/users/views.py         → GoogleLoginCallbackView, UserMeView
django_core/apps/users/serializers.py   → UserSerializer
src/app/login/page.tsx                  → Google 로그인 버튼
src/app/auth/callback/page.tsx          → JWT 수신 + 저장
```

### .env 추가 필요
```env
GOOGLE_CLIENT_ID=...
GOOGLE_CLIENT_SECRET=...
FRONTEND_URL=http://localhost:3000
DJANGO_SECRET_KEY=강한시크릿키
```

### Navbar 로그인 상태 UI
```
비로그인: User 아이콘 → /login 링크
로그인됨: 닉네임 버튼 → 드롭다운 (이메일 표시 + 로그아웃)
```

---

## 🗂 프론트엔드 v2 완성 (Phase 1)

### 신규 파일 (4개)
```
src/lib/score.ts                      → 점수 유틸 (GEM_TIERS, MATCH_TIERS, getScoreGradient)
src/components/ui/ErrorState.tsx      → 공통 에러 UI (network/not-found/generic 자동 감지)
src/components/ui/LoadingSkeleton.tsx → 스켈레톤 (Card/Grid/Ranking/Detail)
src/components/ui/GameImage.tsx       → next/image 최적화 + CDN 폴백 체인
```

### 주요 개선 (9개)
```
GemBadge: 임계값 95 → 70 (v5 점수 체계 기준)
MatchBar: '52' → '52 / 99' 단위 명확화 + compact 모드
useRecommend: useEffect+mutation → useQuery (30분 캐싱)
getDistinctiveMetrics: 값 기준 → 편차 기준 (게임 정체성 표현)
메인 페이지: URL 기반 검색 (Suspense + useSearchParams)
랭킹 페이지: useQuery 전환 + ErrorState + Skeleton
RankingList: img → GameImage
RecommendList: ErrorState + GameGridSkeleton 추가
search/page.tsx: 이중 상태 제거 (mutation.data 직접 사용)
```

### 미사용 패키지 제거
```bash
npm uninstall framer-motion class-variance-authority @radix-ui/react-dialog @radix-ui/react-tabs
# 번들 크기 ~78KB 감소
```

---

## 🚨 트러블슈팅

### 1. npm run build 실패 — useSearchParams Suspense 누락
```
에러: useSearchParams() should be wrapped in a suspense boundary at page "/"
원인: useSearchParams()를 Suspense 없이 사용
해결: HomeContent 컴포넌트 분리 → HomePage에서 <Suspense> 래핑
```

### 2. taste/action created_at null 위반
```
에러: null value in column 'created_at' violates not-null constraint
원인: SQLAlchemy server_default=func.now() 설정했으나
      Django 테이블에 DB DEFAULT 없음 → INSERT에 포함 안 됨
해결: created_at=datetime.now(timezone.utc) Python에서 직접 주입
```

### 3. DEFAULT_PREFERENCES 추천 품질 저하
```
문제: narrative_depth: 8 설정했는데 Spider-Man, GTA, 배트맨 추천
원인: 6개 지표 분산으로 액션-어드벤처 지표도 같이 primary 처리
해결: 핵심 3개 지표만 → 매일 순환 테마로 교체
```

### 4. Sentry DSN 로그 안 뜸
```
원인: logging 설정 없어서 logger.info() 출력 안 됨
      실제로는 정상 작동 중 → /ops/sentry-test로 확인
해결: 로그 없어도 정상, curl 테스트로 검증
```

---

## 💡 인사이트

- **분산 선호도의 함정**: 지표를 여러 개 동시에 지정하면 각각 primary 처리 → 액션 게임에 유리
  → 핵심 3개 지표로 압축하면 변별력 확실히 올라감
- **SSR sessionId**: `generateSessionId()` 초기값으로 쓰면 매 새로고침마다 바뀜
  → null 초기화 + onRehydrateStorage lazy 생성이 올바른 패턴
- **sendBeacon**: 페이지 이탈 시 fetch 요청은 취소될 수 있음
  → Steam 클릭 같은 navigation 이벤트는 반드시 sendBeacon 사용
- **Raw SQL 피하기**: FastAPI에서 Django 테이블 접근 시 raw SQL보다 SQLAlchemy ORM이 일관성 높음
  → Django DEFAULT 동작 차이로 인한 버그도 ORM으로 해결
- **오퍼스 크로스리뷰 효과**: 세션 ID 보안 취약점, SSR 이슈 등 로컬 검토에서 놓친 것들 발견
  → 중요 코드는 다른 Claude 인스턴스에게도 리뷰 받는 게 효과적

---

## 📋 다음 할 일

### 즉시 (Phase 2 마무리)
- [ ] Google Cloud Console OAuth2 클라이언트 ID 발급
- [ ] .env에 GOOGLE_CLIENT_ID, GOOGLE_CLIENT_SECRET 추가
- [ ] Django --build 후 migrate
- [ ] Google 로그인 동작 테스트
- [ ] Django Admin에 Google 소셜 앱 등록

### Phase 2 남은 것
- [ ] 스와이프 온보딩 (로그인 직후)
- [ ] 취향 DNA 카드
- [ ] JWT refresh 자동 갱신 (axios interceptor)

### Phase 3
- [ ] Vercel + Railway 배포
- [ ] 취향 학습 (유저 100명+, 행동 1만 건+)
- [ ] Steam OpenID 연동